# Frameworks de Big Data de Última Generación (2025) + Comparativo de Desempeño

**Objetivo:** Explorar el ecosistema moderno (Spark/Flink/Kafka/NoSQL/Vector DBs/DuckDB/Polars) y ejecutar un comparativo reproducible entre un enfoque clásico y uno moderno en un problema de escala.

---
## 1. Panorama 2025: Marco de Referencia

| Capa | Tecnologías Clave | Casos de uso |
|------|--------------------|--------------|
| Ingesta/Eventos | Kafka, Pulsar, Event Hubs | Telemetría, logs, CDC, IoT |
| Stream Proc. | Flink 1.18+, Spark Structured Streaming | Ventanas, estado, joins en tiempo real |
| Batch Proc. | Spark 3.5+, dbt Core, Ray | Transformaciones, ML feature pipelines |
| Query Serving | DuckDB, Trino, BigQuery, Snowflake | SQL interactivo, BI |
| NoSQL | Cassandra, DynamoDB, MongoDB | Clave-valor, wide-column, documentos |
| Vector DB | Milvus, pgvector, Pinecone | Búsqueda semántica, RAG |
| ML/FS | Feast/Tecton, MLflow | Feature store, model registry |
| Formatos | Parquet, Delta, Iceberg, Hudi | Columnares, ACID, time travel |

---
## 2. Spark vs Flink (visión)

| Aspecto | Spark 3.5+ | Flink 1.18+ |
|---------|------------|-------------|
| Modelo | Micro-batch/continuous | Streaming nativo (event-time, estado) |
| Latencia | Baja (ms–s) | Muy baja (ms) |
| Batch | Excelente | Bueno (Blink planner) |
| APIs | DataFrame, SQL | DataStream, Table/SQL |
| ETL | Muy fuerte (ecosistema) | Fuerte (stateful streaming) |

---
## 3. NoSQL y Bases Vectoriales

- Cassandra/DynamoDB: wide-column/kv para escrituras masivas y latencias estables.
- MongoDB/DocumentDB: flexibilidad de esquema para documentos JSON.
- Vector DBs (Milvus, pgvector, Pinecone): búsquedas por similitud en embeddings (RAG/LLMs).

---
## 4. Comparativo de Desempeño: Clásico vs Moderno

Evaluaremos una tarea típica: agregaciones y join sobre datos sintéticos (~5–10M filas).

- Clásico: pandas (memoria única, single-thread).
- Moderno: Polars (vectorizado, multi-thread) y/o DuckDB (vectorizado + SQL in-process).

Notas:
- Si no están instalados `polars` o `duckdb`, el notebook seguirá con pandas y mostrará advertencias.
- Este comparativo es ilustrativo; en entornos reales use datasets mayores y hardware adecuado.

In [ ]:
import time, random
import pandas as pd

# Intentar importar opciones modernas
try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

N = 1_000_000  # tamaño moderado para ejecutar localmente
random.seed(42)

# Generar dataset sintético
users = pd.DataFrame({
    'user_id': range(N),
    'country': [random.choice(['US','ES','MX','BR','DE','FR']) for _ in range(N)],
    'age': [random.randint(18, 70) for _ in range(N)]
})

events = pd.DataFrame({
    'user_id': [random.randint(0, N-1) for _ in range(N)],
    'event_type': [random.choice(['view','click','purchase']) for _ in range(N)],
    'amount': [random.random()*100 for _ in range(N)]
})

def benchmark_pandas(users: pd.DataFrame, events: pd.DataFrame):
    t0 = time.time()
    # Join
    joined = events.merge(users, on='user_id', how='left')
    # Agregación
    result = joined.groupby(['country','event_type'])['amount'].sum().reset_index()
    dt = time.time()-t0
    return dt, result.head()

def benchmark_polars(users: pd.DataFrame, events: pd.DataFrame):
    if not HAS_POLARS:
        return None, None
    t0 = time.time()
    u = pl.from_pandas(users)
    e = pl.from_pandas(events)
    joined = e.join(u, on='user_id', how='left')
    result = (joined
              .groupby(['country','event_type'])
              .agg(pl.col('amount').sum())
             )
    dt = time.time()-t0
    return dt, result.head()

def benchmark_duckdb(users: pd.DataFrame, events: pd.DataFrame):
    if not HAS_DUCKDB:
        return None, None
    t0 = time.time()
    con = duckdb.connect(':memory:')
    con.register('users', users)
    con.register('events', events)
    query = 

,
,
,
,
,


    res = con.execute(query).fetchdf()
    dt = time.time()-t0
    return dt, res.head()

print('=== Benchmarks (tiempos en segundos) ===')
p_dt, p_head = benchmark_pandas(users, events)
print(f'pandas:   {p_dt:.3f}s')

if HAS_POLARS:
    pl_dt, pl_head = benchmark_polars(users, events)
    print(f'polars:   {pl_dt:.3f}s')
else:
    print('polars:   no instalado (pip install polars)')

if HAS_DUCKDB:
    dd_dt, dd_head = benchmark_duckdb(users, events)
    print(f'duckdb:   {dd_dt:.3f}s')
else:
    print('duckdb:   no instalado (pip install duckdb)')

print('
Muestras de salida:')
print('pandas:
', p_head)
if HAS_POLARS:
    print('polars:
', pl_head)
if HAS_DUCKDB:
    print('duckdb:
', dd_head)

## 5. Interpretación de Resultados

- Polars y DuckDB suelen superar a pandas en operaciones de agregación y joins pesados gracias a ejecución vectorizada y paralelismo.
- En producción, Spark/Flink se usan para datasets distribuidos (TB–PB) y cargas continuas.
- DuckDB y Polars complementan para análisis locales rápidos (exploración, prototipos, CI).

## 6. Guía de Selección (rápida)

- ¿Datos en PB + múltiples equipos? → Spark/Flink + formatos abiertos (Iceberg/Delta).
- ¿SQL local interactivo en archivos Parquet? → DuckDB.
- ¿Transformaciones locales rápidas en DataFrames? → Polars.
- ¿Búsqueda semántica/LLMs? → Vector DB (pgvector/Milvus).

## 7. Buenas Prácticas 2025

- Mantener datos en formatos abiertos (evitar lock-in).
- Definir contrato de esquemas y estrategia de evolución.
- Monitorear costo/latencia con SLOs y auto-escalado.
- Integrar catálogo + linaje + calidad (OpenLineage/Atlas/DataHub).